In [1]:
import pandas as pd
import numpy as np
import requests
import matplotlib.pyplot as plt
import plotly.express as px
from flipside import Flipside
from datetime import datetime, date, timedelta
from dateutil.relativedelta import relativedelta
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
from memory_profiler import profile
import json
import csv
import os
import logging
import sys
from importlib import reload
from pymongo import MongoClient, UpdateOne
from web3 import Web3
import psycopg2
from psycopg2.extras import execute_values
from io import StringIO
from typing import Any, Dict
from keys import KEYS

In [2]:
# logging configurations
reload(logging)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)

In [3]:
reload(logging)

class FixedDateCurrentTimeFormatter(logging.Formatter):
    def formatTime(self, record, datefmt=None):
        # Get the current time but replace the date with 05 October 2024
        log_time = datetime.fromtimestamp(record.created) - timedelta(hours=6)  # Subtract 4 hours
        fixed_time = datetime(2024, 10, 18, log_time.hour, log_time.minute, log_time.second)
        return fixed_time.strftime(datefmt or "%Y-%m-%d %H:%M:%S")

formatter = FixedDateCurrentTimeFormatter("%(asctime)s - %(levelname)s - %(message)s")

handler = logging.StreamHandler()
handler.setFormatter(formatter)

logger = logging.getLogger()
logger.setLevel(logging.INFO)
logger.addHandler(handler)

logger.info("Log message with fixed date and current time.")

2024-10-18 12:03:56 - INFO - Log message with fixed date and current time.


## Extract Raw Data

### Data Extraction from Flipside Crypto

In [4]:
# Initilize Flipside Client
flipside_key = KEYS['flipside_key']
flipside = Flipside(flipside_key, "https://api-v2.flipsidecrypto.xyz")

In [5]:
def format_query(query_path: str, params: dict) -> str:
    try:
        with open(query_path, "r") as file:
            query = file.read()
        formatted_query = query.format(**params)
        return formatted_query
    except Exception as e:
        raise ValueError(f"format query error: {e}")

In [6]:
def createQueryRun(query : str, api_key:str = flipside_key) -> str :
    
    
    url = "https://api-v2.flipsidecrypto.xyz/json-rpc"

    # Request headers
    headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key
        }

    # Request payload
    payload = {
        "jsonrpc": "2.0",
        "method": "createQueryRun",
        "params": [
            {
                "resultTTLHours": 1,
                "maxAgeMinutes": 0,
                "sql": query ,
                "tags": {
                    "source": "postman-demo",
                    "env": "test"
                },
                "dataSource": "snowflake-default",
                "dataProvider": "flipside"
            }
        ],
        "id": 1
    }

    # Submit createQueryRun request
    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        if response.status_code == 200:
            logging.info("Query run created successfully!")
            logging.debug(response.json())  # Output the response
            return  response.json()['result']['queryRequest']['queryRunId']
        else:
            raise requests.exceptions.HTTPError(
                f"Unexpected status code: {response.status_code}. Details: {response.text}" )
    except Exception as e:
        logging.error(f" createQueryRun Error: {e}")
    

In [7]:
def getQueryRun(queryRunId:str , api_key:str = flipside_key) -> str:
    
    url = "https://api-v2.flipsidecrypto.xyz/json-rpc"

    # Request headers
    headers = {
    "Content-Type": "application/json",
    "x-api-key": api_key
        }
    
    payload = {
    "jsonrpc": "2.0",
    "method": "getQueryRun",
    "params": [
        {
            "queryRunId": queryRunId
        }
    ],
    "id": 1
    }

    try:
        response = requests.post(url, json=payload, headers=headers)
        response.raise_for_status()
        logging.debug(f'getQueryRun state {response.json()['result']['queryRun']['state']}')
        return response.json()['result']['queryRun']['state']
        
    except Exception as e:
        logging.error(f" getQueryRun Error: {e}")

In [8]:
def queryresult_Pagination(queryRunId:str, page_size:int = 70000) -> list:
       
    current_page_number = 1
    total_pages = 3

    all_rows = []

    while current_page_number <= total_pages:

        try:
            results = flipside.get_query_results(
                queryRunId,
                page_number=current_page_number,
                page_size=page_size         
            )
       
            if results.records:
                total_pages = results.page.totalPages
                all_rows.extend(results.records)
                logging.debug(f"Current page number: {current_page_number} Total Pages: {total_pages}, Rows Retrieved: {len(results.records)}")
            else: 
                logging.warning('No record')
                break

        except Exception as e:
            logging.error(f" Pagination Error: {e}")
            return None
        
        current_page_number += 1

    logging.info(f"Total Pages: {total_pages}, Rows Retrieved: {len(all_rows)}")
        
        
    return all_rows

In [9]:
def extract_flipsidecrypto_data(query_path:str, params: dict , api_key = flipside_key, retry_time:int = 90 ,timeout:int = 600 ) -> list:
    
    try:
        logging.info(f'Start query with params:{params}')
        query = format_query(query_path,params)
        queryRunId = createQueryRun(query,api_key)

        state = None
        start_time = time.time()

        while state != 'QUERY_STATE_SUCCESS':
            
            state = getQueryRun(queryRunId,api_key)

            if state == 'QUERY_STATE_SUCCESS':
                 break 

            elif state in ['QUERY_STATE_FAILED', 'QUERY_STATE_CANCELED']:
                raise RuntimeError(f"Query execution failed or was canceled. State: {state}")
            
            elif state in ['QUERY_STATE_STREAMING_RESULTS', 'QUERY_STATE_RUNNING', 'QUERY_STATE_READY']:
                if time.time() - start_time > timeout:
                    raise TimeoutError("Query execution exceeded timeout limit.")
                
                logging.info(f"Wainting query excution")
                logging.debug(f"retry after {retry_time} sec")

                time.sleep(retry_time)

            else: raise ValueError(f"Unexpected query state: {state}")

            
        result = queryresult_Pagination(queryRunId)

    except TimeoutError as e:
        logging.error(f"Timeout Error: {e}")
        return None
    except RuntimeError as e:
        logging.error(f"Runtime Error: {e}")
        return None
    except Exception as e:
        logging.error(f" state Error: {e}")
        return None
    
                   

    return result

In [10]:
def get_start_block_number(pool: str, file_path: str, default: int = 0) -> int:
    try:
        with open(file_path, 'r') as file:
            block_numbers = json.load(file)
    except (FileNotFoundError, json.JSONDecodeError) as e:
        logging.error(f"Error reading or parsing file {file_path}: {e}")
        return default  
    except Exception as e:
        logging.error(f"Unexpected error: {e}")
        return default  

    block_number = block_numbers.get(pool, default)
    if isinstance(block_number, int):
        return block_number
    else:
        logging.warning(f"Invalid block number for pool '{pool}': {block_number}. Returning default value: {default}")
        return default

In [11]:
def update_start_block_number(data:list, file_path:str) -> None :
    try:
        try:
            with open(file_path, 'r') as file:
                block_numbers = json.load(file)
        except (FileNotFoundError, json.JSONDecodeError):
            block_numbers = {}

        for pool, events in data.items():
            if events:

                last_block_number = max(event['block_number'] for event in events)

                block_numbers[pool] = last_block_number

                logging.debug(f"Block number for pool {pool} updated to {last_block_number}")
            else:
                logging.warning(f"No events found for pool {pool}, skipping.")

    except Exception as e:
        logging.error(f"Error updating block numbers: {e}")
        
    try:
        
        with open(file_path, 'w') as file:
                json.dump(block_numbers, file, indent=4)
                logging.info(f"block number config File updated")
                
    except Exception as e:
        logging.error(f"Error updating block number config File: {e}")
        
    return None

In [12]:
def fetch_positionsData(pool_address:str, query_path:str, block_number_config_file_path:str) -> list:
    try: 
        block_number = get_start_block_number(pool_address,block_number_config_file_path)
        logging.info(f"querying data for pool : {pool_address} starting from block number: {block_number}")
        position_data = extract_flipsidecrypto_data(query_path, params= {'pool_address': pool_address,'block_number':block_number} )
        logging.info(f"position data fetched successfully for pool {pool_address}, Rows Retrieved: {len(position_data)}")
    except Exception as e:
        logging.error(f"Error fetching position data for pool: {pool_address}: {e}")
    return position_data

In [13]:
def search_pools(query_path:str,numberofpools:int):    

    ### get Top {numberofpools} Pools with higest volume 1 month period

    pool_search_query_params = {'NumberOfpools':numberofpools}
    extracted_pools = extract_flipsidecrypto_data(query_path, pool_search_query_params)
    pools_list = [pool['pool_address'] for pool in extracted_pools]
    
    return pools_list

In [14]:
def fetch_positionData_all_pools(pool_addresses:list,query_path_positionData:str,file_path_block_number:str) -> list:
    all_results = {}
    try:   
        with ThreadPoolExecutor() as executor:
            future_to_pool = {executor.submit(fetch_positionsData, pool,query_path_positionData,file_path_block_number): pool for pool in pool_addresses}
            for future in as_completed(future_to_pool):
                pool = future_to_pool[future]
                result = future.result()
                all_results[pool] = result
    except Exception as e:
        logging.error(f"error in  fetching  positionData: {e}")

    update_start_block_number(all_results,file_path_block_number)
    flattened_values = [item for sublist in all_results.values() for item in sublist]           
    return flattened_values

In [15]:
def normalize_pool_addresses(pool_addresses_list:list) -> tuple:
    try:
        pool_addresses_tuple = tuple(pool_addresses_list)
        if len(pool_addresses_tuple) == 1:
            pool_addresses_tuple = (pool_addresses_tuple[0], pool_addresses_tuple[0])
        return pool_addresses_tuple
    except Exception as e:
        logging.error(f"Error Normalizing addresess list: {e}")
    

In [16]:

pool_search_query_path = r'sql_queries\search_pools_by_volume_query.sql'
pools_list = search_pools(pool_search_query_path,2)


2024-10-18 12:03:56 - INFO - Start query with params:{'NumberOfpools': 2}
2024-10-18 12:03:59 - INFO - Query run created successfully!
2024-10-18 12:04:00 - INFO - Wainting query excution
2024-10-18 12:05:31 - INFO - Total Pages: 1, Rows Retrieved: 2


In [17]:
pools_list

['0x50fe1432a9127b25d81ba12d739b744f84111134',
 '0x914565e885cb9df773e83b0ecbdabb36c0de3b10']

In [18]:
positin_data_query_path = r'sql_queries\get_position_data_query.sql'
block_number_config_file_path = 'configs/block_number_config.json'
positions_extracted_data = fetch_positionData_all_pools(pools_list,positin_data_query_path,block_number_config_file_path)

2024-10-18 12:05:31 - INFO - querying data for pool : 0x914565e885cb9df773e83b0ecbdabb36c0de3b10 starting from block number: 0
2024-10-18 12:05:31 - INFO - querying data for pool : 0x50fe1432a9127b25d81ba12d739b744f84111134 starting from block number: 0
2024-10-18 12:05:31 - INFO - Start query with params:{'pool_address': '0x914565e885cb9df773e83b0ecbdabb36c0de3b10', 'block_number': 0}
2024-10-18 12:05:31 - INFO - Start query with params:{'pool_address': '0x50fe1432a9127b25d81ba12d739b744f84111134', 'block_number': 0}
2024-10-18 12:05:32 - INFO - Query run created successfully!
2024-10-18 12:05:32 - INFO - Query run created successfully!
2024-10-18 12:05:33 - INFO - Wainting query excution
2024-10-18 12:05:33 - INFO - Wainting query excution
2024-10-18 12:07:05 - INFO - Total Pages: 1, Rows Retrieved: 151
2024-10-18 12:07:05 - INFO - position data fetched successfully for pool 0x914565e885cb9df773e83b0ecbdabb36c0de3b10, Rows Retrieved: 151
2024-10-18 12:07:05 - INFO - Total Pages: 1, R

In [19]:
positions_extracted_data

[{'pool_address': '0x914565e885cb9df773e83b0ecbdabb36c0de3b10',
  'block_hash': '0xdc49cbabd7e3da3613ffcbb1d44b72bd7245f1ddd017388830571b6ddc62dc74',
  'block_number': 16928128,
  'tx_hash': '0xb6b12aa0b18d335eeb5c57df62c3bd2138d088350977d5b8d9d32b346e8159f4',
  'tx_index': 163,
  'contract_address': '0xc36442b4a4522e871399cd717abdd847ab11fe88',
  'event_index': 331,
  'block_timestamp': '2023-03-28T20:13:11.000Z',
  'origin_from_address': '0x67c717434d5ec1b0266f9fb384eba3e88f49102d',
  'origin_to_address': '0xc36442b4a4522e871399cd717abdd847ab11fe88',
  'topic0': '0x3067048beee31b25b2f1681f88dac838c8bba36af25bfb2b7cf7473a5847e35f',
  'event_name': 'IncreaseLiquidity',
  'topics': ['0x3067048beee31b25b2f1681f88dac838c8bba36af25bfb2b7cf7473a5847e35f',
   '0x0000000000000000000000000000000000000000000000000000000000074b41'],
  'data': '0x000000000000000000000000000000000000000000000000018f34747cb0b498000000000000000000000000000000000000000000000000000470240a40e000000000000000000000000000

In [20]:
pool_info_query_path = r'sql_queries\get_pool_info_query.sql'
pools_list_tuple = normalize_pool_addresses(pools_list)
pool_info_data = extract_flipsidecrypto_data(pool_info_query_path, params= {'pool_address': pools_list_tuple} )

2024-10-18 12:07:05 - INFO - Start query with params:{'pool_address': ('0x50fe1432a9127b25d81ba12d739b744f84111134', '0x914565e885cb9df773e83b0ecbdabb36c0de3b10')}
2024-10-18 12:07:06 - INFO - Query run created successfully!
2024-10-18 12:07:06 - INFO - Wainting query excution
2024-10-18 12:08:38 - INFO - Total Pages: 1, Rows Retrieved: 2


In [21]:
pool_info_data

[{'pool_address': '0x914565e885cb9df773e83b0ecbdabb36c0de3b10',
  'token0': '0x5ee84583f67d5ecea5420dbb42b462896e7f8d06',
  'token1': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
  'fee': 10000,
  'tickspacing': 200,
  '__row_index': 0},
 {'pool_address': '0x50fe1432a9127b25d81ba12d739b744f84111134',
  'token0': '0x808507121b80c02388fad14726482e061b8da827',
  'token1': '0xc02aaa39b223fe8d0a0e5c4f27ead9083c756cc2',
  'fee': 10000,
  'tickspacing': 200,
  '__row_index': 1}]

### Loading Raw data to MongoDB

In [22]:
def initialize_mongo_db(host, database_name):
    try:
        client = MongoClient(host)
        db = client[database_name]
    except Exception as e:
        logging.error(f"Error in initiating Mongo client: {e}")    

    return client,db

In [23]:
def bulk_upsert_mongodb(host, database_name, collection_name, data, unique_keys):
    try:
        client, db = initialize_mongo_db(host, database_name)
        if client is None or db is None:
            logging.error("Failed to connect to MongoDB.")
            return None
        else: 
            logging.debug('Connected to mongoDB successfully')
            

        collection = db[collection_name]

        
        bulk_operations = [
            UpdateOne(
                {key: record[key] for key in unique_keys if key in record}, 
                {"$setOnInsert": record},
                upsert=True
            )
            for record in data
        ]

        result = None
        if bulk_operations:
            result = collection.bulk_write(bulk_operations, ordered=False)
            logging.debug(f"Inserted: {result.upserted_count}")
            if result.upserted_count > 0:
                logging.info("Mongodb: Data inserted successfully!")
            else: 
                logging.info("Mongodb: No new data to be inserted!")
        return result
    
            
    
    except Exception as e:
        logging.error(f"Error in bulk upsert operation: {e}")
        return None

    finally:
        if client:
            client.close()
            logging.debug("MongoDB: Client closed!") 


In [24]:
def load_config(config_path="configs/mongodb_config.json"):
    with open(config_path, "r") as file:
        return json.load(file)["database"]

In [25]:
config = load_config()
host = config["host"]
database_name = config["name"]
collections = config["collections"]
print(collections)

{'collection_postion_Data': {'collection_name': 'positionsdata_raw', 'unique_keys': ['block_number', 'tx_hash', 'event_index']}, 'collection_pool_info_Data': {'collection_name': 'poolinfo_data', 'unique_keys': ['pool_address']}}


In [26]:
collection_postion_Data = config["collections"]["collection_postion_Data"]["collection_name"]
collection_postion_Data_keys = config["collections"]["collection_postion_Data"]["unique_keys"]
collection_pool_info_Data = config["collections"]["collection_pool_info_Data"]["collection_name"]
collection_pool_info_Data_keys = config["collections"]["collection_pool_info_Data"]["unique_keys"]

In [27]:
# insert position data
upsert_result_position_data = bulk_upsert_mongodb(host,database_name, collection_postion_Data, positions_extracted_data,collection_postion_Data_keys)

2024-10-18 12:08:38 - INFO - Mongodb: Data inserted successfully!


In [28]:
# insert position data
upsert_result_poolinfo_data = bulk_upsert_mongodb(host,database_name, collection_pool_info_Data, pool_info_data,collection_pool_info_Data_keys)

2024-10-18 12:08:38 - INFO - Mongodb: No new data to be inserted!
